# Overview

We are going to use ASE to calculate the thermochemistry of an ideal gas. ASE also contains various routines to calculate the thermochemistry of solids and molecular adsorbates on surfaces, but we will focus on lone molecules for the time being.

For this exercise, we will use a brand new machine learning potential called [UPET](https://github.com/lab-cosmo/upet). This is a machine learning surrogate for DFT, which we will learn about later in the course.


# Setup


In [2]:
!uv pip install upet

Using Python 3.13.7 environment at: /Users/ct5868/personal/cbe423/.venv
Audited 1 package in 17ms


Until a [bug is fixed](https://github.com/lab-cosmo/upet/issues/99), you will need to download the ML model checkpoint yourself from [HuggingFace](https://huggingface.co/lab-cosmo/upet/tree/main/models). Please download `pet-mad-s-v1.5.0.ckpt` and put it in the current working directory. I also tried to automate this below, but there are no guarantees.


In [3]:
!curl -L "https://huggingface.co/lab-cosmo/upet/resolve/main/models/pet-mad-s-v1.5.0.ckpt" -o pet-mad-s-v1.5.0.ckpt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1331  100  1331    0     0  13003      0 --:--:-- --:--:-- --:--:-- 13049
100  105M  100  105M    0     0  13.0M      0  0:00:08  0:00:08 --:--:-- 11.4M84  105M   84 88.9M    0     0  13.6M      0  0:00:07  0:00:06  0:00:01 13.5M


# Demonstration


First, we are going to instantiate the ASE calculator. As a reminder, this tells ASE how the energies and forces will be calculated. It could be any method, such as DFT, but here we will use the faster UPET machine learned interatomic potential.


In [4]:
from upet.calculator import UPETCalculator

calc = UPETCalculator(model="pet-mad-s", version="1.5.0")

/Users/ct5868/personal/cbe423/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Now we are going to pick a system to study. Let's go through this exercise for N2O. Our goal is to go from E (i.e. 0 K electronic energy) to G (i.e. Gibbs free energy).


First, we need to make sure we optimize our structure so that it is a minimum in the potential energy landscape. We do that by first defining our molecule, attaching the calculator to it, and running the optimization. We do not need to wrap the molecule in a `FrechetCellFilter` here because there is no unit cell to optimize.


In [5]:
from ase.build import molecule
from ase.optimize import BFGS

atoms = molecule("N2O")
atoms.calc = calc
opt = BFGS(atoms)
opt.run(fmax=1)

      Step     Time          Energy          fmax
BFGS:    0 14:20:05      -22.871157        4.273861


/Users/ct5868/personal/cbe423/.venv/lib/python3.13/site-packages/metatomic/torch/ase_calculator.py:965: RuntimeWarning: invalid value encountered in scalar add
  (stress[1, 2] + stress[2, 1]) / 2.0,
/Users/ct5868/personal/cbe423/.venv/lib/python3.13/site-packages/metatomic/torch/ase_calculator.py:966: RuntimeWarning: invalid value encountered in scalar add
  (stress[0, 2] + stress[2, 0]) / 2.0,


BFGS:    1 14:20:05      -22.632887       11.609624
BFGS:    2 14:20:05      -22.965738        1.520908
BFGS:    3 14:20:05      -22.972101        0.292071
BFGS:    4 14:20:05      -22.972691        0.078416
BFGS:    5 14:20:05      -22.972731        0.012804
BFGS:    6 14:20:05      -22.972733        0.009531
BFGS:    7 14:20:05      -22.972740        0.007840
BFGS:    8 14:20:05      -22.972744        0.007431
BFGS:    9 14:20:05      -22.972752        0.014818
BFGS:   10 14:20:06      -22.972767        0.028315
BFGS:   11 14:20:06      -22.972816        0.051744
BFGS:   12 14:20:06      -22.972961        0.064441
BFGS:   13 14:20:06      -22.971365        0.452249
BFGS:   14 14:20:06      -22.973101        0.067375
BFGS:   15 14:20:06      -22.973202        0.056463
BFGS:   16 14:20:06      -22.939211        2.174058
BFGS:   17 14:20:06      -22.973248        0.050657
BFGS:   18 14:20:06      -22.973276        0.041417


/Users/ct5868/personal/cbe423/.venv/lib/python3.13/site-packages/metatomic/torch/ase_calculator.py:967: RuntimeWarning: invalid value encountered in scalar add
  (stress[0, 1] + stress[1, 0]) / 2.0,


BFGS:   19 14:20:06      -22.973373        0.048229
BFGS:   20 14:20:06      -22.973396        0.031303
BFGS:   21 14:20:06      -22.973473        0.007508
BFGS:   22 14:20:06      -22.973543        0.046254
BFGS:   23 14:20:06      -22.973568        0.036058
BFGS:   24 14:20:06      -22.973610        0.040806
BFGS:   25 14:20:06      -22.973618        0.016183
BFGS:   26 14:20:06      -22.973625        0.008804
BFGS:   27 14:20:06      -22.973629        0.005309
BFGS:   28 14:20:06      -22.973629        0.001848
BFGS:   29 14:20:06      -22.973629        0.001327
BFGS:   30 14:20:06      -22.973629        0.001437
BFGS:   31 14:20:06      -22.973629        0.001270
BFGS:   32 14:20:06      -22.973629        0.001722
BFGS:   33 14:20:06      -22.973629        0.002598
BFGS:   34 14:20:06      -22.973629        0.003518
BFGS:   35 14:20:06      -22.973633        0.005686
BFGS:   36 14:20:06      -22.973637        0.009366
BFGS:   37 14:20:06      -22.973640        0.015564
BFGS:   38 1

np.True_

Great! We have optimized the geometry. Now, we are going to make sure we fetch the final energy (E), which we will need for later.


In [6]:
E = atoms.get_potential_energy()
print(E)

-22.97369384765625


In [7]:
from ase.visualize import view

view(atoms, viewer="x3d")

To go from an energy to Gibbs free energy, we need the vibrational modes. For that, we must run a vibrational frequency analysis. This has the added benefit of telling us if we are at a minimum in the potential energy landscape as intended.


In [8]:
from ase.vibrations import Vibrations

vib = Vibrations(atoms)
vib.clean()
vib.run()

/Users/ct5868/personal/cbe423/.venv/lib/python3.13/site-packages/metatomic/torch/ase_calculator.py:967: RuntimeWarning: invalid value encountered in scalar add
  (stress[0, 1] + stress[1, 0]) / 2.0,


Note that we should expect there to be 3N-5 vibrational modes for this molecule since it is linear (if it were nonlinear, there would be 3N-6). ASE does not remove any of these spurious modes in the vibrational summary printout.


In [9]:
print(f"There should be {3 * len(atoms) - 5} real vibrational modes.")

There should be 4 real vibrational modes.


In [10]:
vib.summary()

---------------------
  #    meV     cm^-1
---------------------
  0    0.0i      0.2i
  1    0.0       0.1
  2    0.0       0.3
  3    2.0      15.7
  4    3.2      25.7
  5   69.8     563.3
  6   70.7     570.1
  7  159.8    1288.9
  8  278.2    2243.9
---------------------
Zero-point energy: 0.292 eV


We can already see that the zero-point vibrational energy (E_ZPVE) was computed based on the vibrational frequencies. That's convenient, but we have to be careful because not all of thsoe modes are physically relevant.


Now we will go ahead and correct E to G(T, P) using the vibrational analysis. ASE comes with many [thermochemistry](https://ase-lib.org/ase/thermochemistry/thermochemistry.html) modules. You should check out the documentation. It includes all the equations and also is generally useful.

We are going to treat the molecule as an ideal gas and so we will use the `IdealGasThermo` class.


In [11]:
from ase.thermochemistry import IdealGasThermo

We have to pass several arguments to the `IdealGasThermoClass`. The vibrational energies are needed for the vibrational heat capacity. We also need to specify whether it is linear or not to get the correct rotational heat capacity. The `potentialenergy` is the E value that the correction will be added to. The `symmetrynumber` is the number of symmetric rotations that the molecule has (it is often easiest to just look this up, although there are symmetry tools to automate this). The `spin` is the 1/2 \* the number of unpaired electrons, which is 0 here.

Note that the spurious vibrational modes will be "cut" automatically by the `IdealGasThermo` class, so you do not need to worry about that.


In [13]:
vib_energies = vib.get_energies()
igt = IdealGasThermo(
    vib_energies, "linear", potentialenergy=E, atoms=atoms, symmetrynumber=1, spin=0
)

Now we can get various properties. Let's start with the E_ZPVE.


In [14]:
igt.get_ZPE_correction()

np.float64(0.28927189781336293)

Now let's get the enthalpy and entropy. We will need to specify the conditions.


In [15]:
T = 298.15  # K
P = 1e5  # Pa

In [16]:
H = igt.get_enthalpy(T)

Enthalpy components at T = 298.15 K:
Enthalpy components at T = 298.15 K:
E_pot                -22.974 eV
E_ZPE                  0.289 eV
Cv_trans (0->T)        0.039 eV
Cv_rot (0->T)          0.026 eV
Cv_vib (0->T)          0.010 eV
-------------------------------
U                    -22.610 eV
(C_v -> C_p)           0.026 eV
-------------------------------
H                    -22.584 eV


In [17]:
S = igt.get_entropy(T, P)

Entropy components at T = 298.15 K and P = 100000.0 Pa:
{"":15s}{"S":13s}     {"T*S:13s}
S_trans (1 bar)    0.0016174 eV/K        0.482 eV
S_rot              0.0006203 eV/K        0.185 eV
S_elec             0.0000000 eV/K        0.000 eV
S_vib              0.0000455 eV/K        0.014 eV
S (1 bar -> P)    -0.0000000 eV/K       -0.000 eV
-------------------------------------------------
S                  0.0022833 eV/K        0.681 eV


If we'd like, we can calculate G manually.


In [18]:
G_manual = H - T * S
print(G_manual)

-23.2651791717082


We can also just request it directly.


In [19]:
G = igt.get_gibbs_energy(T, P)
print(G)

Enthalpy components at T = 298.15 K:
Enthalpy components at T = 298.15 K:
E_pot                -22.974 eV
E_ZPE                  0.289 eV
Cv_trans (0->T)        0.039 eV
Cv_rot (0->T)          0.026 eV
Cv_vib (0->T)          0.010 eV
-------------------------------
U                    -22.610 eV
(C_v -> C_p)           0.026 eV
-------------------------------
H                    -22.584 eV

Entropy components at T = 298.15 K and P = 100000.0 Pa:
{"":15s}{"S":13s}     {"T*S:13s}
S_trans (1 bar)    0.0016174 eV/K        0.482 eV
S_rot              0.0006203 eV/K        0.185 eV
S_elec             0.0000000 eV/K        0.000 eV
S_vib              0.0000455 eV/K        0.014 eV
S (1 bar -> P)    -0.0000000 eV/K       -0.000 eV
-------------------------------------------------
S                  0.0022833 eV/K        0.681 eV

Free energy components at T = 298.15 K and P = 100000.0 Pa:
    H        -22.584 eV
 -T*S         -0.681 eV
-----------------------
    G        -23.265 eV
-23.26517

Finally, we can conclude by noting what the magnitude of this Gibbs free energy correction was.


In [20]:
print(f"The difference between G and E is {G - E} eV")

The difference between G and E is -0.29148532405195127 eV


This is a fairly large difference. The magnitude of the difference will typically be smaller for solids since they lack translational and rotational entropy contributions, but it can still be important.


Now let's see what happens when we bump up the temperature.


In [21]:
G_1000 = igt.get_gibbs_energy(1000, P, verbose=False)
print(f"The difference between G(1000 K) and G(298.15 K) is {G_1000 - G} eV")

The difference between G(1000 K) and G(298.15 K) is -1.839208627569704 eV
